# 05: Execution-based metrics

This is the headline notebook for the thesis: the metrics here say *did the translation preserve semantics?*, which is the question the whole project is trying to answer. They are also the most expensive metrics to compute, because they actually run queries against three live databases.

**Methodology: what's the oracle?**

Unlike Spider / Text2Cypher benchmarks, which use the *gold Cypher* as the oracle (assuming the human-written reference is correct), we use the **gold SQL on Postgres** as the oracle. This is a stronger setup for our project because:

1. The SQL is the *input*: it's what the user actually wanted answered.
2. The gold Cypher / AQL can be wrong; the SQL can also be wrong, but if the SQL is wrong the whole row in the dataset is bad and the user would have caught it.
3. It tests *end-to-end* semantic preservation: Postgres on the relational side vs Neo4j / ArangoDB on the graph side.

**Pipeline per record (LDBC only, TPC-H Postgres isn't loaded):**

1. Run the gold SQL on graphonauts2's Postgres → reference rows.
2. If the translation `validation_passed`, run the *generated query* on the appropriate graph DB → translated rows.
3. Normalise both row sets (canonical tuples of strings: dates as ISO, numbers as repr, nulls as `None`).
4. Compare as multisets: execution accuracy (equality), precision, recall, F1, Jaccard.

**Skips:**
- TPC-H records: no Postgres backing exists. Static metrics still apply to them.
- Records with `validation_passed == False` : there's no syntactically-valid query to execute. Their EX/F1 default to 0.0 / NaN.
- Records whose graph DB execution times out (default 30 s) or errors: `execution_error` is recorded, EX/F1 = 0.0.

**Caveat:** result-set comparison is positional. The translated query must return the same *columns in the same order* as the SQL. Column-name mismatches (`liker.id` vs `liker_id`) don't matter; column-order mismatches DO. We pre-aligned the gold queries' RETURN clauses so positional comparison is fair.

In [ ]:
from __future__ import annotations

import json
import os
from collections import Counter
from datetime import date, datetime
from pathlib import Path
from time import perf_counter
from typing import Any

import pandas as pd
import psycopg
from arango.client import ArangoClient
from neo4j import GraphDatabase
from neo4j.time import Date as Neo4jDate, DateTime as Neo4jDateTime

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

OUTPUTS_DIR = REPO_ROOT / "evaluation" / "outputs"
RECORDS_PATH = OUTPUTS_DIR / "records.json"
OUT_CSV = OUTPUTS_DIR / "metrics_execution.csv"
CACHE_PATH = OUTPUTS_DIR / "execution_rows_cache.json"

STATEMENT_TIMEOUT_S = int(os.environ.get("EVAL_QUERY_TIMEOUT", "30"))
print(f"Per-query timeout: {STATEMENT_TIMEOUT_S}s")

## Connection configs

Defaults match `graphonauts2/docker/*.compose.yml` and `config/servers/*.yaml`. Override via environment variables if your setup differs.

In [ ]:
PG_DSN = (
    f"host={os.environ.get('POSTGRES_HOST', 'localhost')} "
    f"port={os.environ.get('POSTGRES_PORT', '5433')} "
    f"user={os.environ.get('POSTGRES_USER', 'graphonaut')} "
    f"password={os.environ.get('POSTGRES_PASSWORD', 'password')} "
    f"dbname={os.environ.get('POSTGRES_DB', 'graphonaut')}"
)
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_DB = os.environ.get("NEO4J_DATABASE", "neo4j")
ARANGO_URL = os.environ.get("ARANGO_URL", "http://localhost:8529")
ARANGO_USER = os.environ.get("ARANGO_USER", "root")
ARANGO_DB = os.environ.get("ARANGO_DATABASE", "ldbc")

for var in ("NEO4J_PASSWORD", "ARANGO_PASSWORD"):
    assert os.environ.get(var), f"Export {var} before running this notebook."

## Row normalisation

Different drivers hand us values in different Python types: Postgres returns native `datetime`, Neo4j returns its own `neo4j.time.DateTime`, ArangoDB returns ISO strings. We canonicalise every value to a string so multiset comparison works across drivers.

Rules:

- `None` → `None` (preserved as sentinel: Postgres NULL and Cypher null both arrive here).
- `bool` → `'True'` / `'False'`.
- Numeric types: `int(x)` if it equals `int(x)`, else `repr(float(x))` rounded to 6 decimals (driver float precision varies).
- Dates / datetimes / Neo4j datetimes: ISO 8601 prefix `YYYY-MM-DD` (date-only) or `YYYY-MM-DDTHH:MM:SS` (datetime). Time zones stripped to avoid `+00:00` vs `Z` mismatches.
- Everything else → `str(x)`.

In [ ]:
def _norm_value(v: Any) -> Any:
    if v is None:
        return None
    if isinstance(v, bool):
        return "True" if v else "False"
    if isinstance(v, (Neo4jDate, Neo4jDateTime)):
        return v.iso_format().split("+")[0].split("[")[0]
    if isinstance(v, datetime):
        return v.replace(tzinfo=None).isoformat(timespec="seconds")
    if isinstance(v, date):
        return v.isoformat()
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        as_int = int(v)
        if abs(v - as_int) < 1e-9:
            return str(as_int)
        return f"{v:.6f}"
    s = str(v)
    # Strings that look like timestamps with timezone: strip the offset.
    if len(s) >= 19 and s[4] == "-" and s[7] == "-" and (s[10] == "T" or s[10] == " "):
        return s.replace(" ", "T")[:19]
    return s


def _norm_row(row: list[Any] | tuple[Any, ...] | dict[str, Any]) -> tuple[Any, ...]:
    if isinstance(row, dict):
        # Stable key order = insertion order of the source query's RETURN clause.
        return tuple(_norm_value(v) for v in row.values())
    return tuple(_norm_value(v) for v in row)


_norm_row({"id": 1, "first_name": "Mahinda", "creation_date": datetime(2010, 6, 1, 12, 30)})

## Multiset-comparison metrics

Given two lists of rows (`reference`, `translated`), build a `Counter` over each and compute:

- **Execution Accuracy (EX)**: `1` if `Counter(ref) == Counter(translated)` else `0`.
- **Precision**: overlap / |translated|.
- **Recall**: overlap / |reference|.
- **F1**: harmonic mean.
- **Jaccard**: `1 - overlap / (|ref ∪ trans|)`.

In [ ]:
def compare_rowsets(ref_rows: list, trans_rows: list) -> dict[str, float]:
    ref = Counter(_norm_row(r) for r in ref_rows)
    trans = Counter(_norm_row(r) for r in trans_rows)
    overlap = sum((ref & trans).values())
    n_ref = sum(ref.values())
    n_trans = sum(trans.values())
    union = sum((ref | trans).values())
    ex = 1.0 if ref == trans else 0.0
    precision = overlap / n_trans if n_trans else (1.0 if n_ref == 0 else 0.0)
    recall = overlap / n_ref if n_ref else (1.0 if n_trans == 0 else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    jaccard = 1.0 - (overlap / union) if union else 0.0
    return {
        "execution_accuracy": ex,
        "result_precision": precision,
        "result_recall": recall,
        "result_f1": f1,
        "result_jaccard_dist": jaccard,
        "reference_rows": n_ref,
        "translated_rows": n_trans,
    }


# Quick check:
compare_rowsets([(1, "a"), (2, "b")], [(2, "b"), (1, "a")])

## Query executors

Three thin wrappers, one per backend. Each returns `(rows, runtime_seconds, error_or_None)`. Errors are caught and surfaced as strings so the loop can record the failure without crashing the whole notebook.

In [ ]:
def run_postgres(sql: str) -> tuple[list[tuple], float, str | None]:
    t0 = perf_counter()
    try:
        with psycopg.connect(PG_DSN) as conn, conn.cursor() as cur:
            cur.execute(f"SET LOCAL statement_timeout = {STATEMENT_TIMEOUT_S * 1000}")
            cur.execute(sql)
            rows = cur.fetchall()
    except Exception as exc:
        return [], perf_counter() - t0, f"{type(exc).__name__}: {exc}"
    return rows, perf_counter() - t0, None


_neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, os.environ["NEO4J_PASSWORD"]))


def run_cypher(query: str) -> tuple[list[dict], float, str | None]:
    t0 = perf_counter()
    try:
        with _neo4j_driver.session(database=NEO4J_DB) as session:
            result = session.run(query, timeout=STATEMENT_TIMEOUT_S)
            rows = [r.data() for r in result]
    except Exception as exc:
        return [], perf_counter() - t0, f"{type(exc).__name__}: {exc}"
    return rows, perf_counter() - t0, None


_arango_client = ArangoClient(hosts=ARANGO_URL)
_arango_db = _arango_client.db(ARANGO_DB, username=ARANGO_USER, password=os.environ["ARANGO_PASSWORD"])


def run_aql(query: str) -> tuple[list, float, str | None]:
    t0 = perf_counter()
    try:
        cursor = _arango_db.aql.execute(query, max_runtime=STATEMENT_TIMEOUT_S)
        rows = list(cursor)
    except Exception as exc:
        return [], perf_counter() - t0, f"{type(exc).__name__}: {exc}"
    return rows, perf_counter() - t0, None

## Reference-row cache

Running the same SQL on Postgres every time the notebook is re-executed would be wasteful (and on hard LDBC queries, slow). We cache reference rows by `query_id`. The cache lives next to `records.json` in `evaluation/outputs/`.

Delete `execution_rows_cache.json` to force a refresh.

In [ ]:
cache: dict[str, dict] = {}
if CACHE_PATH.exists():
    cache = json.loads(CACHE_PATH.read_text())
print(f"Reference cache: {len(cache)} entries.")

## Main loop

Per LDBC record:

1. Fetch (or cache) reference rows from Postgres.
2. If `validation_passed`, run the generated query on the appropriate graph DB.
3. Compare row multisets.
4. Append result.

In [ ]:
records = json.loads(RECORDS_PATH.read_text())
ldbc_records = [r for r in records if r["schema_name"] == "ldbc"]
print(f"LDBC records: {len(ldbc_records)} (skipping {len(records) - len(ldbc_records)} non-LDBC).")

rows_out: list[dict] = []
for idx, rec in enumerate(ldbc_records, start=1):
    qid = rec["query_id"]
    print(f"[{idx:3d}/{len(ldbc_records)}] {qid} ({rec['dialect']})", end=" ", flush=True)
    if qid in cache:
        ref_rows = [tuple(r) for r in cache[qid]["rows"]]
        ref_runtime = cache[qid]["runtime"]
        ref_error = cache[qid].get("error")
    else:
        ref_rows, ref_runtime, ref_error = run_postgres(rec["sql"])
        cache[qid] = {"rows": [list(r) for r in ref_rows], "runtime": ref_runtime, "error": ref_error}
        CACHE_PATH.write_text(json.dumps(cache, default=str))

    out_row: dict[str, Any] = {
        "query_id": qid,
        "schema_name": rec["schema_name"],
        "dialect": rec["dialect"],
        "difficulty": rec["difficulty"],
        "validation_passed": rec["validation_passed"],
        "reference_runtime_s": ref_runtime,
        "reference_error": ref_error,
        "translated_runtime_s": None,
        "execution_error": None,
        "execution_accuracy": 0.0,
        "result_precision": 0.0,
        "result_recall": 0.0,
        "result_f1": 0.0,
        "result_jaccard_dist": 1.0,
        "reference_rows": len(ref_rows),
        "translated_rows": 0,
    }
    if ref_error is not None:
        print(f"REF ERROR ({ref_error})")
        rows_out.append(out_row); continue
    if not rec["validation_passed"] or not rec.get("generated_query"):
        print("skip (translation invalid)")
        rows_out.append(out_row); continue

    runner = run_cypher if rec["dialect"] == "cypher" else run_aql
    trans_rows, trans_runtime, trans_error = runner(rec["generated_query"])
    out_row["translated_runtime_s"] = trans_runtime
    out_row["execution_error"] = trans_error
    if trans_error is not None:
        print(f"EXEC ERROR ({trans_error[:60]})")
        rows_out.append(out_row); continue

    metrics = compare_rowsets(ref_rows, trans_rows)
    out_row.update(metrics)
    marker = "✓" if metrics["execution_accuracy"] == 1.0 else "≠"
    print(
        f"{marker} EX={metrics['execution_accuracy']:.0f} F1={metrics['result_f1']:.2f} "
        f"ref={metrics['reference_rows']} trans={metrics['translated_rows']} "
        f"({ref_runtime:.2f}s ref / {trans_runtime:.2f}s gen)"
    )
    rows_out.append(out_row)

exec_df = pd.DataFrame(rows_out)
print("\nDone.")

## Summaries

In [ ]:
metric_cols = ["execution_accuracy", "result_precision", "result_recall", "result_f1", "result_jaccard_dist"]
print("Overall:")
display(exec_df[metric_cols].mean().to_frame("mean"))
print("\nBy dialect:")
display(exec_df.groupby("dialect")[metric_cols].mean())
print("\nBy difficulty:")
display(exec_df.groupby("difficulty")[metric_cols].mean().reindex(["easy", "medium", "hard"]))
print("\nExecution errors:")
errs = exec_df[exec_df["execution_error"].notna()][["query_id", "dialect", "execution_error"]]
display(errs)

In [ ]:
exec_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(exec_df)} rows to {OUT_CSV}")

In [ ]:
# Release driver resources.
_neo4j_driver.close()
print("Drivers closed.")